# Generating shrunken logFC (posterior-mean) matrices

Turns a perturb-seq screen into a shrunken log-fold-change matrix in which **each
perturbation appears several times** — once from all its cells, and once per subsample of
them. Useful when you want many logFC estimates per perturbation at varying precision.

Pipeline (scripts under `src/module_finder/de/`):

| step | script | what it does |
|---|---|---|
| 1 | `ComputeSE.py` | cells → per-label `mean_diff` + Welch `se`, chunked |
| 2 | `run_ashr_on_chunk.R` | `ash(mean_diff, se)` per chunk |
| 3 | `MergeAshrRes.py` | merge label/gene metadata back in |
| 4 | `AshrGenerateMatrix.py` | pivot to wide label × gene `PosteriorMean` |

**Subsampling.** Perturbations with more than 50 cells get `N_SUBSAMPLES` subsamples whose
sizes span `[50, n_cells)`; the cells in each are drawn uniformly without replacement.
Perturbations at or below 50 cells contribute their full row only.

**Controls.** A fixed set of `N_CONTROL_CELLS` controls is drawn once and shared by every
row, so the control side of the standard error is identical across rows.

Every knob is a plain variable in the config cell, passed straight to the functions.

Requires R with the `ashr` package.

In [4]:
import gc
import shutil
import sys
import time
from pathlib import Path

PROJECT = Path.cwd().parents[1]          # notebooks/module_finder -> project root
sys.path.insert(0, str(PROJECT / "src"))

import numpy as np
import pandas as pd
import anndata as ad

from module_finder.de import (
    generate_posterior_matrices,
    plan_augmentation,
    build_augmented_adata,
    subsample_sizes,
)

## 1. Point at your data

You need one `.h5ad` per context, each with:

- log-normalised expression in `.X` (or name a `layer`),
- a `.obs` column of perturbation labels,
- a control label within that column.

A *context* is one condition (here, one drug); its file name becomes the output name. The
walkthrough below uses a single context, then the loop in section 5 runs them all.

Every knob is a plain variable here, passed straight to the functions. Note the two
different floors: `MIN_CELLS_TO_SUBSAMPLE` (50) decides which perturbations are *eligible
for subsampling*, while `MIN_CELLS` (20) is `ComputeSE.py`'s cutoff for dropping a
perturbation altogether.

In [3]:
# ---------------- input: one .h5ad per context ----------------
# Full objects: every gene (18,154), full control depth (~300k cells/drug).
# This is the source the published PosteriorMeanMatrices/ were built from.
SCREEN_DIR = Path("/processed_datasets/VCI/ChemoGenetic_H1_Basak")

# Listed explicitly, not globbed: the directory also holds Round-1 contexts
# (CHIR, DMSO, KYA, LDN, PFI1, RGFP) that are not part of this 16-context screen.
CONTEXTS = [
    "DMSO_round2", "DMSO_round2_batch2",          # vehicle controls
    "AR-A014418", "AZD4573", "Bisindolylmaleimide-I", "CHIR-98014", "DG-172",
    "JTE-607", "LDN-193189", "LY2090314", "Lexibulin", "NSC95397", "PP121",
    "Romidepsin", "Stattic", "VX-11e",
]

def context_path(context):
    return SCREEN_DIR / context / f"{context}.h5ad"

PERTURBATION_KEY = "target_gene"
CONTROL_LABEL = "non-targeting"
LAYER = None                    # None uses .X

# ---------------- subsampling knobs ----------------
N_SUBSAMPLES = 6                # subsamples per eligible perturbation
SPACING = "linear"              # "log" covers precision more evenly (se ~ 1/sqrt(n))
MIN_CELLS_TO_SUBSAMPLE = 50     # only subsample perturbations with more cells than this
N_CONTROL_CELLS = 10_000        # fixed control set, shared by every row
MIN_CELLS = 20                  # ComputeSE.py drops perturbations below this entirely

# ---------------- run scope ----------------
SUBSET_PERTS = None             # int for a quick pass, None for every perturbation
SUBSET_GENES = None             # int for a quick pass, None for every gene
MATRIX_FORMAT = "parquet"       # far smaller than csv at this width
N_ASHR_JOBS = 16                # parallel R workers for the ashr step
DELETE_INTERMEDIATES = False    # keep chunk_*.csv so a re-run skips ComputeSE.py

# /large_storage has room to spare, so intermediates are worth keeping.
OUT_DIR = Path("/large_storage/ctc/beraslan/ModuleFinder/posterior_matrices")
OUT_DIR.mkdir(parents=True, exist_ok=True)

missing = [c for c in CONTEXTS if not context_path(c).exists()]
total_gb = sum(context_path(c).stat().st_size for c in CONTEXTS if context_path(c).exists()) / 1e9
print(f"{len(CONTEXTS)} contexts in {SCREEN_DIR}")
print(f"  {total_gb/1000:.2f} TB of input, {total_gb/len(CONTEXTS):.0f} GB per context on average")
if missing:
    print("  MISSING:", missing)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/path/to/your/screen.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
# Survey every context before committing compute. Backed reads, so this is cheap.
def survey(context):
    a = ad.read_h5ad(context_path(context), backed="r")
    counts = a.obs[PERTURBATION_KEY].value_counts()
    perts = counts.drop(CONTROL_LABEL, errors="ignore")
    eligible = perts[perts > MIN_CELLS_TO_SUBSAMPLE]
    rows = len(perts) + sum(
        len(subsample_sizes(int(n), N_SUBSAMPLES, MIN_CELLS_TO_SUBSAMPLE, SPACING))
        for n in eligible
    )
    return {"context": context, "cells": a.n_obs, "genes": a.n_vars,
            "perturbations": len(perts), "controls": int(counts.get(CONTROL_LABEL, 0)),
            "eligible": len(eligible), "median_cells": int(perts.median()),
            "matrix_rows": rows}

overview = pd.DataFrame([survey(c) for c in CONTEXTS]).set_index("context")
print(overview.to_string())
print(f"\ntotal matrix rows across the screen: {overview.matrix_rows.sum():,}")

## 2. Check the subsample sizes

`subsample_sizes` is pure arithmetic — see what you will get before touching the data.
The full row sits at `n_cells`, so the subsamples stop just below it and together they
span the range.

In [ ]:
for n_cells in (60, 300, 2000, 10000):
    print(f"{n_cells:>6} cells -> "
          f"{subsample_sizes(n_cells, N_SUBSAMPLES, MIN_CELLS_TO_SUBSAMPLE, SPACING)}"
          f" + full at {n_cells}")

## 3. Inspect the plan

`plan_augmentation` decides every output row before any expression is read, so the rule
is auditable on its own.

In [ ]:
def load_context(context):
    """Read one context, optionally subset, and bring it into memory."""
    a = ad.read_h5ad(context_path(context), backed="r")
    if SUBSET_PERTS is None and SUBSET_GENES is None:
        return a.to_memory()
    counts = a.obs[PERTURBATION_KEY].value_counts()
    keep = [CONTROL_LABEL] + [p for p in counts.index if p != CONTROL_LABEL][:SUBSET_PERTS]
    mask = a.obs[PERTURBATION_KEY].isin(keep).to_numpy()
    genes = slice(None) if SUBSET_GENES is None else slice(0, SUBSET_GENES)
    return a[mask, genes].to_memory()


# walk through one context; the loop further down does all of them
CONTEXT = CONTEXTS[0]
adata = load_context(CONTEXT)
print(CONTEXT, adata.shape)

perturbations = adata.obs[PERTURBATION_KEY].astype(str).to_numpy()

cell_index, labels, plan = plan_augmentation(
    perturbations,
    control_label=CONTROL_LABEL,
    n_subsamples=N_SUBSAMPLES,
    min_cells_to_subsample=MIN_CELLS_TO_SUBSAMPLE,
    n_control_cells=N_CONTROL_CELLS,
    spacing=SPACING,
    seed=0,
)

print(f"{len(plan)} planned rows from {adata.n_obs:,} cells "
      f"-> {len(cell_index):,} cells after duplication\n")
print(plan["variant"].value_counts().to_string())
plan.head(12)

In [ ]:
full = plan[plan.variant == "full"].set_index("perturbation")["n_cells_planned"]
sub = plan[plan.variant == "subsample"]
too_small = full[full <= MIN_CELLS_TO_SUBSAMPLE].index

print("perturbations at/below the floor:", len(too_small))
print("  none subsampled              :", not sub.perturbation.isin(too_small).any())
print("  every n >= floor             :", bool((sub.n_cells_planned >= MIN_CELLS_TO_SUBSAMPLE).all()))
print("  no subsample equals full size:",
      bool((sub.n_cells_planned < full.reindex(sub.perturbation).to_numpy()).all()))
print("  control set fixed at         :",
      int(plan.loc[plan.variant == "control", "n_cells_planned"].item()), "cells")

per_pert = sub.groupby("perturbation").n_cells_planned.agg(["count", "min", "max"])
print(f"\nsubsamples per perturbation: median {int(per_pert['count'].median())} "
      f"of {N_SUBSAMPLES} requested")
per_pert.head()

## 4. Build the augmented object

Full-data cells keep their perturbation label; subsampled cells are **duplicated** under
`<pert>__sub<k>`, which is what lets one `ComputeSE.py` pass produce both variants — and
puts them in one `ashr` fit, where their differing precisions belong. Controls are never
duplicated.

This cell is optional: step 5 does it for you. Run it to see the size cost.

In [ ]:
augmented, plan = build_augmented_adata(
    adata,
    perturbation_key=PERTURBATION_KEY,
    control_label=CONTROL_LABEL,
    n_subsamples=N_SUBSAMPLES,
    min_cells_to_subsample=MIN_CELLS_TO_SUBSAMPLE,
    n_control_cells=N_CONTROL_CELLS,
    spacing=SPACING,
    seed=0,
)
print(f"{adata.n_obs:,} cells -> {augmented.n_obs:,} ({augmented.n_obs / adata.n_obs:.2f}x)")
print(augmented.obs["mf_label"].value_counts().head(8).to_string())

# Section 5 builds this again internally, so drop it here rather than holding two copies
# of a multi-million-cell object.
del augmented
gc.collect();

## 5. Run it

One call covers the whole chain: build the augmented object, write it, run all four
scripts, collect the results.

`chunk_perts` sets how many labels share an `ashr` fit. Raise it above the row count if
you want a single prior over every row.

First one context, then all of them.

**Scale.** These are the full objects: 18,154 genes, 1.6–2.8M cells, 125–242 GB each —
36M cells and 2.96 TB across the 16 contexts, giving ~224,000 matrix rows in total
(surveyed). Per context, expect roughly

- ~14,000 matrix rows (~2,350 perturbations, ~1,950 of them eligible for subsampling),
- ~254M observations through `ashr`,
- ~150 GB resident for the source object and ~1.9x that once augmented, so
  **run one context at a time**,
- a transient augmented `.h5ad` of similar size to the input, plus ~60 GB of
  intermediates that are kept by default.

A smoke run on 25 perturbations x 400 genes took ~2 min. This is orders of magnitude
larger — run the single-context cell first and read its reported timing before starting
the loop.

In [ ]:
run = generate_posterior_matrices(
    adata,
    context=CONTEXT,
    out_dir=OUT_DIR,
    perturbation_key=PERTURBATION_KEY,
    control_label=CONTROL_LABEL,
    layer=LAYER,
    n_subsamples=N_SUBSAMPLES,
    min_cells_to_subsample=MIN_CELLS_TO_SUBSAMPLE,
    n_control_cells=N_CONTROL_CELLS,
    spacing=SPACING,
    seed=0,
    min_cells=MIN_CELLS,
    chunk_perts=100,
    n_ashr_jobs=N_ASHR_JOBS,
    matrix_format=MATRIX_FORMAT,
)
print(run)
print("\nmatrix       :", run.matrix_path)
print("row metadata :", run.row_metadata_path)
print("ashr version :", run.meta["ashr_version"])
print("build %.1fs, pipeline %.1fs" % (run.meta["build_seconds"], run.meta["pipeline_seconds"]))

### All contexts

Loops over every context, skipping any already finished so the cell is resumable, and
carrying on past a failure so one bad context does not lose the rest.

Intermediates (~26 GB of `chunk_*.csv` and `AshrResult_chunk_*.csv` per context at full
width) are kept by default since the output volume has room, which makes a re-run skip
`ComputeSE.py`. Set `DELETE_INTERMEDIATES = True` if you would rather not keep ~400 GB.

In [ ]:
def run_context(context, permute=False):
    a = load_context(context)
    try:
        r = generate_posterior_matrices(
            a,
            context=context,
            out_dir=OUT_DIR,
            perturbation_key=PERTURBATION_KEY,
            control_label=CONTROL_LABEL,
            layer=LAYER,
            n_subsamples=N_SUBSAMPLES,
            min_cells_to_subsample=MIN_CELLS_TO_SUBSAMPLE,
            n_control_cells=N_CONTROL_CELLS,
            spacing=SPACING,
            seed=0,
            permute=permute,
            permute_seed=0,
            min_cells=MIN_CELLS,
            chunk_perts=100,
            n_ashr_jobs=N_ASHR_JOBS,
            matrix_format=MATRIX_FORMAT,
        )
    finally:
        del a
        gc.collect()
    if DELETE_INTERMEDIATES:
        for pattern in ("chunk_*.csv", "AshrResult_chunk_*.csv"):  # ~26 GB/context
            for f in r.out_dir.glob(pattern):
                f.unlink()
    return r

summary, failures = [], {}

for context in CONTEXTS:
    done = OUT_DIR / context / f"PosteriorMean_matrix_{context}.{MATRIX_FORMAT}"
    if done.exists():
        print(f"[skip] {context:<22} already done")
        continue

    t0 = time.time()
    try:
        r = run_context(context)
        summary.append({
            "context": context,
            "rows": r.matrix.shape[0],
            "genes": r.matrix.shape[1],
            "cells_in": r.meta["n_input_cells"],
            "cells_augmented": r.meta["n_augmented_cells"],
            "minutes": round((time.time() - t0) / 60, 1),
            "matrix": r.matrix_path,
        })
        print(f"[ok]   {context:<22} {r.matrix.shape} in {summary[-1]['minutes']:.1f} min")
        # keep memory flat across 16 contexts -- the matrices are on disk
        del r
        gc.collect()
    except Exception as exc:                     # carry on; inspect failures after
        failures[context] = exc
        print(f"[FAIL] {context:<22} {type(exc).__name__}: {exc}")

    free = shutil.disk_usage(OUT_DIR).free / 1e12
    print(f"       {free:.1f} TB free")
    if free < 0.5:
        print("       stopping: less than 500 GB left")
        break

summary = pd.DataFrame(summary)
if len(summary):
    print(f"\n{len(summary)} contexts done, "
          f"{summary.minutes.sum():.0f} min total, "
          f"{summary.rows.sum():,} rows\n")
    print(summary.drop(columns="matrix").to_string(index=False))
if failures:
    print("\nfailed:", ", ".join(failures))

## 6. Results

`run.matrix` is labels × genes: `<pert>` rows are full-data estimates, `<pert>__sub<k>`
rows are subsamples. `run.row_metadata` says how many cells produced each row.

In [ ]:
print(run.matrix.shape)
run.matrix.iloc[:6, :5]

In [ ]:
run.row_metadata.head(10)

In [ ]:
meta = run.row_metadata.set_index("label")
subs = meta[meta.variant == "subsample"]

rows = []
for label, r in subs.iterrows():
    if r.perturbation in run.matrix.index:
        a, b = run.matrix.loc[r.perturbation], run.matrix.loc[label]
        rows.append({
            "perturbation": r.perturbation,
            "n_sub": int(r.n_cells_used),
            "frac_cells": r.n_cells_used / meta.loc[r.perturbation, "n_cells_used"],
            "corr_to_full": np.corrcoef(a, b)[0, 1],
            "shrinkage_ratio": np.abs(b).mean() / max(np.abs(a).mean(), 1e-12),
        })
concordance = pd.DataFrame(rows)
print(concordance[["frac_cells", "corr_to_full", "shrinkage_ratio"]].describe().round(3).to_string())
concordance.head()

In [ ]:
# fewer cells -> larger se -> ashr shrinks harder
ax = concordance.plot.scatter("frac_cells", "shrinkage_ratio", alpha=.6,
                              title="subsampled rows are shrunk more")
ax.set_xlabel("fraction of cells used"); ax.set_ylabel("|PosteriorMean| relative to full");

## 7. A label-permuted null (optional)

Same call with `permute=True`: perturbation labels are shuffled among the perturbed
cells, controls untouched, every label keeping its cell count.

It is a null for **specificity**, not for overall effect size — the average perturbation
effect still reaches every row, because the control baseline is left intact.

In [ ]:
null_run = generate_posterior_matrices(
    adata,
    context=CONTEXT,
    out_dir=OUT_DIR,
    perturbation_key=PERTURBATION_KEY,
    control_label=CONTROL_LABEL,
    layer=LAYER,
    n_subsamples=N_SUBSAMPLES,
    min_cells_to_subsample=MIN_CELLS_TO_SUBSAMPLE,
    n_control_cells=N_CONTROL_CELLS,
    spacing=SPACING,
    seed=0,
    permute=True,
    permute_seed=0,
    min_cells=MIN_CELLS,
    chunk_perts=100,
    n_ashr_jobs=N_ASHR_JOBS,
    matrix_format=MATRIX_FORMAT,
)
print(null_run)

real, null = run.matrix.std(axis=0).mean(), null_run.matrix.std(axis=0).mean()
print(f"\nmean across-row SD per gene: real {real:.5f} | permuted {null:.5f} "
      f"| ratio {real / max(null, 1e-12):.2f}x")
print("(near 1 means no more perturbation-specific structure than the null)")

## 8. What landed on disk

```
/large_storage/ctc/beraslan/ModuleFinder/posterior_matrices/<CONTEXT>/
    PosteriorMean_matrix_<CONTEXT>.csv   labels x genes, full + subsampled rows
    row_metadata.csv                     label -> perturbation, variant, n_cells_used
    module_finder_manifest.json          parameters, timings, ashr version
    chunk_*.csv                          ComputeSE.py output (mean_diff, se)
    AshrResult_chunk_*.csv               ashr output, metadata merged back in
```

With `DELETE_INTERMEDIATES = False` the intermediates stay and a re-run skips
`ComputeSE.py`; with it `True` only the matrix, row metadata and manifest survive, and a
re-run recomputes from cells.

In [ ]:
skip = ("chunk_", "AshrResult_chunk_")
for context_dir in sorted(d for d in OUT_DIR.iterdir() if d.is_dir()):
    kept = [f for f in sorted(context_dir.iterdir())
            if f.is_file() and not f.name.startswith(skip)]
    total = sum(f.stat().st_size for f in context_dir.rglob("*") if f.is_file())
    print(f"{context_dir.name}  ({total/1e9:.1f} GB incl. intermediates)")
    for f in kept:
        print(f"    {f.stat().st_size/1e6:9.2f} MB  {f.name}")